# Imputation-Techniques Benchmark — ALL methods, resumable (saves as it goes)

Runs **every** imputation method in the registry (~40, including the deep family:
GP-VAE, BRITS, M-RNN, GRU-D, SAITS, Transformer, TimesNet, CSDI, US-GAN) on the three
networks (**Dhaka, Beijing, Delhi**) by reconstruction quality: hide a sample of
observed test cells, reconstruct, score RMSE/MAE/R² + imputability at the hidden cells.

### ⭐ It saves after every single method and resumes after a disconnect
Results are written to **Google Drive**, one file per method
(`<RESULTS_DIR>/<dataset>/per_method/<method>.csv`), **the instant that method finishes**.
If Colab disconnects, just re-run the **Run** cell — it skips every method already on disk
(no re-training, no re-fitting) and continues from where it stopped. A drop costs you at
most the single method that was in flight.

### How to use
1. **Runtime → Change runtime type → GPU** (deep models need it).
2. Run **Setup** (upload `air-transformer.zip`), **Install**, **Prepare data**.
3. Run **Mount Drive + Config**, then the **Run** cell.
4. If disconnected at any point: reconnect and just re-run **Run** — it resumes.
5. When all methods are done, run **Aggregate → Figures → Download**.

## 1. Setup — upload the repo zip

In [ ]:
import os

def _find_repo():
    for c in ('.', 'air-transformer'):
        if os.path.exists(os.path.join(c, 'config.yaml')):
            return c
    return None

if _find_repo() is None:
    from google.colab import files
    print('Upload air-transformer.zip (config.yaml must be at its root) ...')
    files.upload()
    os.system('unzip -q -o air-transformer.zip')

repo = _find_repo()
assert repo is not None, 'config.yaml not found — did the zip extract correctly?'
os.chdir(repo)
print('Working dir:', os.getcwd())

## 2. Install the extra imputers (don't touch Colab's CUDA torch)

In [ ]:
import torch
print('Torch:', torch.__version__,
      '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE — set Runtime > Change runtime type > GPU')
# A wheel that fails just demotes its method to "skipped" — the run continues.
!pip install -q pypots fancyimpute scikit-fuzzy minisom
print('Extras installed.')

## 3. Prepare data — clean all three networks
Beijing/Delhi auto-download; resumes from existing files.

In [ ]:
!python scripts/01_prepare_data.py  --config config.yaml
!python scripts/01b_prepare_beijing.py --config config_beijing.yaml
!python scripts/01c_prepare_delhi.py   --config config_delhi.yaml
print('Processed parquets + scalers ready.')

## 4. Mount Google Drive + configuration

`RESULTS_DIR` lives on Drive so partial results **survive disconnects**. Set
`USE_DRIVE = False` to keep results on the (ephemeral) Colab disk instead.

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/imputation_benchmark'
else:
    RESULTS_DIR = 'outputs/imputation_benchmark'
os.makedirs(RESULTS_DIR, exist_ok=True)

SMOKE_TEST   = True            # True = ~2-min wiring check (Dhaka, 5 fast methods)
HOLDOUT_RATE = 0.20
PATTERNS     = ('mcar', 'outage')
SEED         = 42
INCLUDE_SLOW = True            # ARIMA, Kalman, SSA, MissForest, SVR, SOM, fuzzy
INCLUDE_DEEP = True            # AE/DAE + all pypots deep models (needs GPU)

DATASETS = [('dhaka', 'config.yaml'),
            ('beijing', 'config_beijing.yaml'),
            ('delhi', 'config_delhi.yaml')]
print('RESULTS_DIR =', RESULTS_DIR)

## 5. Run — builds + scores + **saves each method as it finishes**

Re-run this cell after any disconnect; it skips everything already saved and resumes.
Every line `saved <method>` means that method's CSV is safely on Drive.

In [ ]:
import warnings, json, yaml
warnings.filterwarnings('ignore')
import torch
from src import imputation_benchmark as ib

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ib.coverage_map().to_csv(os.path.join(RESULTS_DIR, 'coverage_map.csv'), index=False)

ds_list = DATASETS[:1] if SMOKE_TEST else DATASETS
only    = {'forward_fill', 'mean', 'linear_interp', 'hour_mean', 'knn'} if SMOKE_TEST else None

for name, cfgpath in ds_list:
    cfg = yaml.safe_load(open(cfgpath))
    proc = cfg['paths']['processed_dir']
    sc = json.load(open(os.path.join(proc, 'scalers.json')))
    print(f'\n========== {name.upper()} (device={device}) ==========', flush=True)
    ib.run_resumable(
        cfg, sc, RESULTS_DIR,
        patterns=PATTERNS, holdout_rate=HOLDOUT_RATE, seed=SEED, device=device,
        include_slow=INCLUDE_SLOW and not SMOKE_TEST,
        include_deep=INCLUDE_DEEP and not SMOKE_TEST,
        only=only,
        max_stations=2 if SMOKE_TEST else None,
        max_slice_len=2000 if SMOKE_TEST else None,
        log=lambda m: print(m, flush=True))

board_all = ib.aggregate_all(RESULTS_DIR)
print('\nAll done. rows (method x dataset x pattern):', len(board_all))

### Leaderboards
`forward_fill` must read `imputability == 0.0` (it is the baseline). `imputability > 0`
means the method beats forward-fill at reconstructing hidden cells.

In [ ]:
import pandas as pd
pd.set_option('display.width', 240); pd.set_option('display.max_columns', 20)
cols = ['method','family','pm25_rmse_ugm3','overall_std_rmse','overall_r2','imputability','runtime_s']
for name in board_all['dataset'].str.lower().unique():
    sub = board_all[(board_all.dataset.str.lower()==name) & (board_all.pattern=='mcar')]
    print(f'\n===== {name.upper()} — MCAR holdout {HOLDOUT_RATE:.0%} =====')
    print(sub.sort_values('overall_std_rmse')[cols].to_string(index=False))

## 6. Figures

In [ ]:
import matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_style('whitegrid')
except Exception:
    sns = None
figdir = os.path.join(RESULTS_DIR, 'figures'); os.makedirs(figdir, exist_ok=True)

for name in board_all['dataset'].str.lower().unique():
    sub = board_all[(board_all.dataset.str.lower()==name) & (board_all.pattern=='mcar')]
    sub = sub.sort_values('overall_std_rmse')
    fams = sub['family'].astype('category')
    colors = plt.cm.tab20(fams.cat.codes / max(1, fams.cat.categories.size))
    fig, ax = plt.subplots(figsize=(8, 0.32*len(sub)+1))
    ax.barh(sub['method'], sub['overall_std_rmse'], color=colors)
    ax.axvline(1.0, ls='--', c='k', lw=1, label='forward-fill')
    ax.set_xlabel('overall standardized RMSE (lower = better)')
    ax.set_title(f'{name.capitalize()} — reconstruction RMSE by method (MCAR)')
    ax.invert_yaxis(); ax.legend(); fig.tight_layout()
    fig.savefig(f'{figdir}/rmse_bar_{name}.png', dpi=150); plt.show()

In [ ]:
piv = (board_all[board_all.pattern=='mcar']
       .pivot_table(index='method', columns='dataset', values='imputability'))
if piv.shape[1] >= 1 and len(piv):
    fig, ax = plt.subplots(figsize=(1.6*piv.shape[1]+3, 0.33*len(piv)+1))
    if sns is not None:
        sns.heatmap(piv.sort_values(piv.columns[0]), annot=True, fmt='.2f',
                    center=0, cmap='RdYlGn', ax=ax, cbar_kws={'label':'imputability'})
    else:
        im = ax.imshow(piv.values, cmap='RdYlGn'); fig.colorbar(im)
    ax.set_title('Imputability vs forward-fill (>0 = beats ffill)')
    fig.tight_layout(); fig.savefig(f'{figdir}/imputability_heatmap.png', dpi=150); plt.show()

## 7. Download a zip of everything

In [ ]:
import shutil
parent, leaf = os.path.split(RESULTS_DIR.rstrip('/'))
z = shutil.make_archive('imputation_benchmark_artifacts', 'zip',
                        root_dir=parent, base_dir=leaf)
print('Wrote', z)
print('Contains every per_method/*.csv, leaderboard_all.csv, coverage_map.csv, figures/.')
try:
    from google.colab import files; files.download(z)
except Exception as e:
    print('(Not in Colab — copy manually):', e)